In [ ]:
import requests
import pandas as pd

class CryptoScreener:
    
    # 1. THE CONSTRUCTOR (The Setup Crew)
    def __init__(self, limit=10):
        # We make the URL dynamic so the user can ask for 10, 50, or 100 assets!
        self.api_url = f"https://api.coingecko.com/api/v3/coins/markets?vs_currency=usd&order=market_cap_desc&per_page={limit}&page=1"
        
        # We create empty placeholders for our DataFrames so the object owns them permanently
        self.market_df = None
        self.trading_df = None

    # 2. THE EXTRACTION METHOD
    def fetch_data(self):
        print(f"Establishing secure connection to CoinGecko API...")
        response = requests.get(self.api_url)
        
        if response.status_code == 200:
            print("Connection Secured. Extracting payload...")
            raw_data = response.json()
            # Storing the raw data permanently to the object using 'self'
            self.market_df = pd.DataFrame(raw_data)
            return True
        else:
            print(f"Critical Failure. Status Code: {response.status_code}")
            return False

    # 3. THE TRANSFORMATION METHOD
    def apply_quantitative_filter(self):
        # Failsafe: Ensure data exists before trying to filter it
        if self.market_df is None:
            print("Error: No data available. Run fetch_data() first.")
            return

        print("Applying quantitative filters and engineering features...")
        
        # The .copy() prevents Pandas from throwing a warning when you slice a DataFrame
        self.trading_df = self.market_df[['symbol', 'name', 'current_price', 'market_cap']].copy()
        
        self.trading_df['Price_Tier'] = "Standard"
        # Notice your fat-finger typo is fixed here (1000.0 instead of 10000)!
        self.trading_df.loc[self.trading_df['current_price'] > 1000.0, 'Price_Tier'] = "Premium"
        
        print("Pipeline complete. Data is ready for the trading algorithm.")

    # 4. THE MASTER TRIGGER
    def run_pipeline(self):
        # This single method chains the other methods together automatically
        if self.fetch_data():
            self.apply_quantitative_filter()
            return self.trading_df